In [ ]:
import logging
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import dask
import dask.array as da
from dask.diagnostics import ProgressBar

In [ ]:
PROJECT_ROOT = Path.cwd().parent.parent
INPUT_DIR    = PROJECT_ROOT / "data/in/era5_daily"
OUTPUT_DIR   = PROJECT_ROOT / "data/out"

YEARS           = range(1950, 1952)   # (start: inclusuve, end: exclusive -> ends 31/12 year -1)
YEAR_START      = YEARS.start
YEAR_END        = YEARS.stop - 1
PERC_YEAR_START = 1965                # percentile reference period start
PERC_YEAR_END   = 1994                # percentile reference period end (inclusive)

OUT_PERC    = OUTPUT_DIR / "percentiles"
OUT_DAILY   = OUTPUT_DIR / "daily"
OUT_MONTHLY = OUTPUT_DIR / "monthly"
OUT_YEARLY  = OUTPUT_DIR / "yearly"
for path in [OUT_PERC, OUT_DAILY, OUT_MONTHLY, OUT_YEARLY]:
    path.mkdir(parents=True, exist_ok=True)

dask.config.set(scheduler='threads')

In [ ]:
# File list
files = sorted(INPUT_DIR.glob("era5_daily_*.nc"))
# Lazy opening
ds = xr.open_mfdataset(files, combine='by_coords', engine="netcdf4")
# Slicing
ds = ds.sel(valid_time=slice(f"{YEAR_START}-01-01", f"{YEAR_END}-12-31"))

# --- DEBUG SINGLE PIXEL (remove for full run) ---
_lat = [45.9]   # ~northern Italy
_lon = [8.6]    # passing lists keeps the lat/lon dimensions (size 1)
ds = ds.sel(latitude=_lat, longitude=_lon, method='nearest')
# --- END DEBUG ---

In [ ]:
ds

In [ ]:
ds["t_mean"].values

In [ ]:
# Reload datasets deleted at end of previous blocks (ds is still open from the top)
temp_perc     = xr.open_dataset(OUT_PERC  / "temp_percentiles.nc", chunks="auto")
wet_days_perc = xr.open_dataset(OUT_PERC  / "precip_percentiles.nc")
daily_ds     = xr.open_mfdataset(sorted(OUT_DAILY.glob("daily_vars_*.nc")))
DTR_d         = daily_ds[['DTR_d']]
PW_d          = daily_ds[['PW_d']]
P_jm          = xr.open_mfdataset(sorted(OUT_MONTHLY.glob("monthly_vars_*.nc")))[['P_jm']]

# --- DEBUG SINGLE PIXEL (remove for full run) ---
temp_perc     = temp_perc.sel(latitude=_lat, longitude=_lon, method='nearest')
wet_days_perc = wet_days_perc.sel(latitude=_lat, longitude=_lon, method='nearest')
DTR_d         = DTR_d.sel(latitude=_lat, longitude=_lon, method='nearest')
PW_d          = PW_d.sel(latitude=_lat, longitude=_lon, method='nearest')
P_jm          = P_jm.sel(latitude=_lat, longitude=_lon, method='nearest')

In [ ]:
P_jm

In [ ]:
TM = (
    ds['t_mean']
    .resample(valid_time='YE')
    .mean(dim='valid_time', skipna=True)
    .rename({'valid_time': 'year'})
    .assign_coords(year=np.unique(ds['valid_time'].dt.year.values))
    .assign_attrs(long_name='Mean annual temperature', units='Celsius')
    .to_dataset(name='TM')
)
TM_computed = TM.compute()
TM_computed

In [ ]:
TX = (
    ds['t_max']
    .resample(valid_time='YE')
    .mean(dim='valid_time', skipna=True)
    .rename({'valid_time': 'year'})
    .assign_coords(year=np.unique(ds['valid_time'].dt.year.values))
    .assign_attrs(long_name='Mean annual maximum temperature', units='Celsius')
    .to_dataset(name='TX')
)
TX_computed = TX.compute()
TX_computed

In [ ]:
TN = (
    ds['t_min']
    .resample(valid_time='YE')
    .mean(dim='valid_time', skipna=True)
    .rename({'valid_time': 'year'})
    .assign_coords(year=np.unique(ds['valid_time'].dt.year.values))
    .assign_attrs(long_name='Mean annual minimum temperature', units='Celsius')
    .to_dataset(name='TN')
)
TN_computed = TN.compute()
TN_computed

In [ ]:
ds_TM = TM
days_per_year = ds['t_mean'].groupby('valid_time.year').count()
TVAR = (
    (
        ((ds['t_mean'].groupby('valid_time.year') - ds_TM['TM']) ** 2)
        .groupby('valid_time.year')
        .sum(skipna=True)
        / (days_per_year - 1)
    )
    .where(ds_TM['TM'].notnull())
).assign_attrs(long_name='Annual temperature variance', units='Celsius^2').to_dataset(name='TVAR')
TVAR_computed = TVAR.compute()
TVAR_computed

In [ ]:
ds_DTR_d = DTR_d
DTR = (
    ds_DTR_d['DTR_d']
    .resample(valid_time='YE')
    .mean(dim='valid_time', skipna=True)
    .rename({'valid_time': 'year'})
    .assign_coords(year=np.unique(ds_DTR_d['valid_time'].dt.year.values))
    .assign_attrs(long_name='Mean annual temperature range', units='Celsius')
    .to_dataset(name='DTR')
)
DTR_computed = DTR.compute()
DTR_computed

In [ ]:
years = np.unique(ds['valid_time'].dt.year.values)
TNN = (
    ds['t_min']
    .resample(valid_time='YE')
    .min(dim='valid_time', skipna=True)
    .rename({'valid_time': 'year'})
    .rename('TNN')
    .assign_coords(year=years)
    .assign_attrs(long_name='Annual minimum of daily minimum temperature', units='Celsius')
)
TNN_computed = TNN.compute()
TNN_computed

In [ ]:
TXX = (
    ds['t_max']
    .resample(valid_time='YE')
    .max(dim='valid_time', skipna=True)
    .rename({'valid_time': 'year'})
    .rename('TXX')
    .assign_coords(year=years)
    .assign_attrs(long_name='Annual maximum of daily maximum temperature', units='Celsius')
)
TXX_computed = TXX.compute()
TXX_computed

In [ ]:
ds_T_pkd_5w = temp_perc

# Remove Feb 29: the percentile thresholds are indexed 1-365, so Feb 29 has no valid threshold
feb29 = (ds['valid_time'].dt.month == 2) & (ds['valid_time'].dt.day == 29)
ds_here = ds.sel(valid_time=~feb29)

# Map each day's normalized day-of-year to the corresponding percentile threshold.
# For leap year dates after Feb 28, dt.dayofyear is 1 higher than the equivalent
# non-leap year date; subtracting 1 gives a consistent DOY 1-365 for the threshold lookup.
is_leap_and_late = ds_here['valid_time'].dt.is_leap_year & (ds_here['valid_time'].dt.month > 2)
doy = ds_here['valid_time'].dt.dayofyear - is_leap_and_late.astype(int)
# For each day in the time series, look up the corresponding percentile threshold
# by its day-of-year. doy is an array of integers (1-365), one per timestep.
# .sel(day_of_year=doy) maps each day timestep throughout the years to its threshold
# producing arrays of shape (time, latitude, longitude) aligned with the daily data.
tn10 = ds_T_pkd_5w['TN_10p_5w'].sel(day_of_year=doy)
tx10 = ds_T_pkd_5w['TX_10p_5w'].sel(day_of_year=doy)
tn90 = ds_T_pkd_5w['TN_90p_5w'].sel(day_of_year=doy)
tx90 = ds_T_pkd_5w['TX_90p_5w'].sel(day_of_year=doy)

years = np.unique(ds_here['valid_time'].dt.year.values)

CN10 = (
    (ds_here['t_min'] < tn10).resample(valid_time='YE').sum()
    .rename({'valid_time': 'year'}).assign_coords(year=years)
    .drop_attrs()
    .assign_attrs(long_name='Number of cold nights (t_min < TN_10p_5w)', units='days')
    .rename('CN10')
)
CN10_computed = CN10.compute()
CN10_computed

In [ ]:
CD10 = (
    (ds_here['t_max'] < tx10).resample(valid_time='YE').sum()
    .rename({'valid_time': 'year'}).assign_coords(year=years)
    .drop_attrs()
    .assign_attrs(long_name='Number of cold days (t_max < TX_10p_5w)', units='days')
    .rename('CD10')
)
CD10_computed = CD10.compute()
CD10_computed

In [ ]:
WN90 = (
    (ds_here['t_min'] > tn90).resample(valid_time='YE').sum()
    .rename({'valid_time': 'year'}).assign_coords(year=years)
    .drop_attrs()
    .assign_attrs(long_name='Number of warm nights (t_min > TN_90p_5w)', units='days')
    .rename('WN90')
)
WN90_computed = WN90.compute()
WN90_computed

In [ ]:
WD90 = (
    (ds_here['t_max'] > tx90).resample(valid_time='YE').sum()
    .rename({'valid_time': 'year'}).assign_coords(year=years)
    .drop_attrs()
    .assign_attrs(long_name='Number of warm days (t_max > TX_90p_5w)', units='days')
    .rename('WD90')
)
WD90_computed = WD90.compute()
WD90_computed

In [ ]:
ds_TX90p15w = temp_perc

# Remove Feb 29 and rechunk to one complete year per time block.
# After removing Feb 29, every year has exactly 365 days, so chunk(valid_time=365) produces one block per year
feb29 = (ds['valid_time'].dt.month == 2) & (ds['valid_time'].dt.day == 29)
ds_here = ds[['t_max']].sel(valid_time=~feb29).chunk({'valid_time': 365})

# Get an array of years present in the data
unique_years = np.unique(ds_here['valid_time'].dt.year.values)
# Get the number of years
n_years = len(unique_years)
# Rechunk the threshold table to match the spatial chunking of ds_here.
# day_of_year must be a single chunk (-1) so that map_blocks passes all 365 thresholds to every block.
tx90_table = ds_TX90p15w['TX_90p_15w'].chunk({'day_of_year': -1, 'latitude': ds_here.chunksizes['latitude'], 'longitude': ds_here.chunksizes['longitude']})

# tx90_block (percentiles) and ds_block (daily values to be evaluated) should be aligned
def process_block(ds_block, tx90_block):

    # Get the time steps in this block
    valid_time = ds_block['valid_time']

    # Normalize DOY to 1-365: leap-year dates after Feb 28 would be 1 day ahead
    # of the equivalent non-leap date, so subtract 1 to keep the threshold lookup consistent
    is_leap_and_late = valid_time.dt.is_leap_year & (valid_time.dt.month > 2)
    doy = (valid_time.dt.dayofyear - is_leap_and_late.astype(int)).values # shape (365,)

    # Extract tmax for this block
    tx = ds_block['t_max'].values # shape (n_days, n_lat, n_lon) - n_days is a multiple of 365
    # Extract the percentiles for this block
    tx90_arr = tx90_block.values # (365, n_lat, n_lon)
    # Get the number of latitude and longitudes coordinates
    n_lat, n_lon = tx.shape[1], tx.shape[2]

    # Output arrays, initialised to NaN
    DDHW = np.full((1, n_lat, n_lon), np.nan, dtype=np.float32)  # total heatwave days
    LDHW = np.full((1, n_lat, n_lon), np.nan, dtype=np.float32)  # longest heatwave (days)
    NDHW = np.full((1, n_lat, n_lon), np.nan, dtype=np.float32)  # number of heatwave events
    TDHW = np.full((1, n_lat, n_lon), np.nan, dtype=np.float32)  # mean temp during heatwaves

    # Loop through each location
    for i in range(n_lat):
        for j in range(n_lon):
            # Get all the tmax data throughout the year for that location
            tx_1d = tx[:, i, j]
            # If everything is NA (ocean cells) continue
            if np.all(np.isnan(tx_1d)):
                continue
            # Look up the daily 90th-percentile threshold for each day of this year at that grid point
            tx90_1d = tx90_arr[doy - 1, i, j]  # doy is 1-based, so subtract 1 for python 0-based indexing
            # Creates binary variables: 1 where t_max exceeds the daily 90th percentile, 0 otherwise
            hot = (tx_1d > tx90_1d).astype(np.int8)
            # Pad with zeros so diff detects runs that start on day 0 or end on the last day
            padded = np.concatenate([[0], hot, [0]])
            # Take the diff between consecutive days, +1 marks the start of a hot spell; -1 marks the end
            d = np.diff(padded)
            # Get all the start dates (value = +1)
            starts = np.where(d == 1)[0]
            # Get all the end dates (value = -1)
            ends = np.where(d == -1)[0]
            # Get the length of heatwaves
            lengths = ends - starts
            # A heatwave requires at least 3 consecutive hot days, so mask
            hw_mask = lengths >= 3
            # Assign 0 values if no heatwave is detected
            if not hw_mask.any():
                DDHW[0, i, j] = 0
                LDHW[0, i, j] = 0
                NDHW[0, i, j] = 0
                continue
            # Use the mask to get the start and the end date of the valid heatwaves only
            hw_starts  = starts[hw_mask]
            hw_lengths = lengths[hw_mask]
            # Calculate the variables
            DDHW[0, i, j] = hw_lengths.sum() # total days across all heatwave events
            LDHW[0, i, j] = hw_lengths.max() # duration of the longest single event
            NDHW[0, i, j] = hw_mask.sum() # count of distinct events
            # Concatenate the actual t_max values from all heatwave periods to compute mean
            hw_tx = np.concatenate([tx_1d[s:s + l] for s, l in zip(hw_starts, hw_lengths)])
            # Calculates mean
            TDHW[0, i, j] = hw_tx.mean()

    # Wrap results back into a Dataset with the first timestamp of the block as the time coordinate
    coords = {'valid_time': ds_block['valid_time'].values[[0]],
              'latitude': ds_block['latitude'].values,
              'longitude': ds_block['longitude'].values}
    dims = ['valid_time', 'latitude', 'longitude']
    return xr.Dataset({
        'DDHW': xr.DataArray(DDHW, dims=dims, coords=coords),
        'LDHW': xr.DataArray(LDHW, dims=dims, coords=coords),
        'NDHW': xr.DataArray(NDHW, dims=dims, coords=coords),
        'TDHW': xr.DataArray(TDHW, dims=dims, coords=coords),
    })

# Get day-1 of the year coordinates (to be assigned to the output coordinates)
year_first_dates = ds_here['valid_time'].values.reshape(n_years, 365)[:, 0]
# Gets spatial chunk sizes in main dataframe (to be given to the output df)
lat_chunks = ds_here.chunksizes['latitude']
lon_chunks = ds_here.chunksizes['longitude']


# Creates the empty (so far) output dataset with the correct structure
template = xr.Dataset({
    var: xr.DataArray(
        da.full(
            (n_years, len(ds_here['latitude']), len(ds_here['longitude'])),
            np.nan, chunks=(1, lat_chunks, lon_chunks), dtype=np.float32,
        ),
        dims=['valid_time', 'latitude', 'longitude'],
        coords={'valid_time': year_first_dates, 'latitude': ds_here['latitude'], 'longitude': ds_here['longitude']},
    )
    for var in ['DDHW', 'LDHW', 'NDHW', 'TDHW']
})

# Runs (lazily) the computations over each chunk
result = xr.map_blocks(process_block, ds_here, args=[tx90_table], template=template)

# Corrects and assigns the correct dimension
result = result.rename({'valid_time': 'year'}).assign_coords(year=unique_years)

# Assigns attributes to each variable
attrs = {
    'DDHW': ('Number of days in day heatwave events (TX > TX_90p_15w for >= 3 consecutive days)', 'days'),
    'LDHW': ('Number of days in the longest day heatwave event of the year', 'days'),
    'NDHW': ('Number of day heatwave events in the year', 'count'),
    'TDHW': ('Average TX during day heatwave days', 'Celsius'),
}
for var, (long_name, unit) in attrs.items():
    result[var].attrs.update(long_name=long_name, units=unit)

result_computed = result.compute()
result_computed

In [ ]:
result_computed["LDHW"]

In [ ]:
ds_TN10p5w = temp_perc

# Remove Feb 29 and rechunk to one complete year per time block.
# After removing Feb 29, every year has exactly 365 days, so chunk(valid_time=365) produces one block per year
feb29 = (ds['valid_time'].dt.month == 2) & (ds['valid_time'].dt.day == 29)
ds_here = ds[['t_min']].sel(valid_time=~feb29).chunk({'valid_time': 365})

# Get an array of years present in the data
unique_years = np.unique(ds_here['valid_time'].dt.year.values)
# Get the number of years
n_years = len(unique_years)
# Rechunk the threshold table to match the spatial chunking of ds, so map_blocks aligns blocks correctly.
# day_of_year must be a single chunk (-1) so that map_blocks passes all 365 thresholds to every block.
tn10_table = ds_TN10p5w['TN_10p_5w'].chunk({'day_of_year': -1, 'latitude': ds_here.chunksizes['latitude'], 'longitude': ds_here.chunksizes['longitude']})

# tn10_block (percentiles) and ds_block (daily values to be evaluated) should be aligned
def process_block(ds_block, tn10_block):

    # Get the time steps in this block
    valid_time = ds_block['valid_time']

    # Normalize DOY to 1-365: leap-year dates after Feb 28 would be 1 day ahead
    # of the equivalent non-leap date, so subtract 1 to keep the threshold lookup consistent
    is_leap_and_late = valid_time.dt.is_leap_year & (valid_time.dt.month > 2)
    doy = (valid_time.dt.dayofyear - is_leap_and_late.astype(int)).values # shape (365,)

    # Extract tmin for this block
    tn = ds_block['t_min'].values # shape (n_days, n_lat, n_lon) - n_days is a multiple of 365
    # Extract the percentiles for this block
    tn10_arr = tn10_block.values # (365, n_lat, n_lon)
    # Get the number of latitude and longitudes coordinates
    n_lat, n_lon = tn.shape[1], tn.shape[2]

    # Output array, initialised to NaN
    CSD = np.full((1, n_lat, n_lon), np.nan, dtype=np.float32)  # total days in cold spells

    # Loop through each location
    for i in range(n_lat):
        for j in range(n_lon):
            # Get all the tmin data throughout the year for that location
            tn_1d = tn[:, i, j]
            # If everything is NA (ocean cells) continue
            if np.all(np.isnan(tn_1d)):
                continue
            # Look up the daily 10th-percentile threshold for each day of this year at that grid point
            tn10_1d = tn10_arr[doy - 1, i, j]  # doy is 1-based, so subtract 1 for python 0-based indexing
            # Creates binary variables: 1 where t_min falls below the daily 10th percentile, 0 otherwise
            cold = (tn_1d < tn10_1d).astype(np.int8)
            # Pad with zeros so diff detects runs that start on day 0 or end on the last day
            padded = np.concatenate([[0], cold, [0]])
            # Take the diff between consecutive days, +1 marks the start of a cold spell; -1 marks the end
            d = np.diff(padded)
            # Get all the start dates (value = +1)
            starts = np.where(d == 1)[0]
            # Get all the end dates (value = -1)
            ends = np.where(d == -1)[0]
            # Get the length of each spell
            lengths = ends - starts
            # A cold spell requires at least 6 consecutive cold days, so mask
            spell_mask = lengths >= 6
            # Sum the total days across all valid cold spells (0 if none)
            CSD[0, i, j] = lengths[spell_mask].sum() if spell_mask.any() else 0

    # Wrap results back into a Dataset with the first timestamp of the block as the time coordinate
    coords = {'valid_time': ds_block['valid_time'].values[[0]],
              'latitude': ds_block['latitude'].values,
              'longitude': ds_block['longitude'].values}
    return xr.Dataset({'CSD': xr.DataArray(CSD, dims=['valid_time', 'latitude', 'longitude'],
                                           coords=coords)})

# Get day-1 of the year coordinates (to be assigned to the output coordinates)
year_first_dates = ds_here['valid_time'].values.reshape(n_years, 365)[:, 0]
# Gets spatial chunks sizes in main dataframe (to be given to the output df)
lat_chunks = ds_here.chunksizes['latitude']
lon_chunks = ds_here.chunksizes['longitude']
# Creates the empty (so far) output dataset with the correct structure
template = xr.Dataset({
    'CSD': xr.DataArray(
        da.full(
            (n_years, len(ds_here['latitude']), len(ds_here['longitude'])),
            np.nan, chunks=(1, lat_chunks, lon_chunks), dtype=np.float32,
        ),
        dims=['valid_time', 'latitude', 'longitude'],
        coords={'valid_time': year_first_dates, 'latitude': ds_here['latitude'], 'longitude': ds_here['longitude']},
    )
})

# Runs (lazily) the computations over each chunk
result = xr.map_blocks(process_block, ds_here, args=[tn10_table], template=template)

# Corrects and assigns the correct dimension
result = result.rename({'valid_time': 'year'}).assign_coords(year=unique_years)

# Assigns attributes to each variable
attrs = {
    'CSD': ('Number of days in cold spell events (TN < TN_10p_5w for >= 6 consecutive days)', 'days'),
}
for var, (long_name, unit) in attrs.items():
    result[var].attrs.update(long_name=long_name, units=unit)

result_computed = result.compute()
result_computed

In [ ]:
PA = (
    ds['tot_prec']
    .resample(valid_time='YE')
    .mean(dim='valid_time', skipna=True)
    .rename({'valid_time': 'year'})
    .assign_coords(year=np.unique(ds['valid_time'].dt.year.values))
    .assign_attrs(long_name='Mean annual precipitation', units='m')
    .to_dataset(name='PA')
)
PA_computed = PA.compute()
PA_computed

In [ ]:
PT = (
    ds['tot_prec']
    .resample(valid_time='YE')
    .sum(dim='valid_time', skipna=True)
    .rename({'valid_time': 'year'})
    .assign_coords(year=np.unique(ds['valid_time'].dt.year.values))
    .assign_attrs(long_name='Mean annual precipitation', units='m')
    .to_dataset(name='PT')
)
PT_computed = PT.compute()
PT_computed

In [ ]:
ds_orig_here = ds
ds_here = PW_d

# True for cells that have at least one non-NaN tot_prec value (i.e. land cells)
land_mask = ds_orig_here['tot_prec'].notnull().any(dim='valid_time')

# PW_d is NaN on both dry days and ocean cells, so an all-NaN year is ambiguous:
# it could mean "no wet days this year" (land) or "no data at all" (ocean).
# min_count=0 resolves this by returning 0 for any all-NaN group; the land_mask
# then restores NaN specifically for ocean cells.
PWT = (
    ds_here['PW_d']
    .resample(valid_time='YE')
    .sum(dim='valid_time', skipna=True, min_count=0)
    .where(land_mask)
    .rename({'valid_time': 'year'})
    .assign_coords(year=np.unique(ds_here['valid_time'].dt.year.values))
    .assign_attrs(long_name='Total annual wet day precipitation', units='m')
    .to_dataset(name='PWT')
)

PWT_computed = PWT.compute()
PWT_computed

In [ ]:
ds_orig_here = ds
ds_here = PW_d

# True for cells that have at least one non-NaN tot_prec value (i.e. land cells)
land_mask = ds_orig_here['tot_prec'].notnull().any(dim='valid_time')

# PW_d is NaN on both dry days and ocean cells, so count() alone cannot distinguish
# "no wet days this year" (land, should be 0) from "no data at all" (ocean, should be NaN).
# count() naturally returns 0 for all-NaN groups; the land_mask then restores NaN
# specifically for ocean cells.
W = (
    ds_here['PW_d']
    .resample(valid_time='YE')
    .count(dim='valid_time')
    .where(land_mask)
    .rename({'valid_time': 'year'})
    .assign_coords(year=np.unique(ds_here['valid_time'].dt.year.values))
    .assign_attrs(long_name='Number of wet days', units='count')
    .to_dataset(name='W')
)
W_computed = W.compute()
W_computed

In [ ]:
ds_PWT = PWT
ds_W = W
# Division is intentionally 0/0 for land cells with no wet days, producing NaN —
# the average is undefined when there are no wet days to average over.
PWA = (
    (ds_PWT['PWT'] / ds_W['W'])
    .assign_attrs(long_name='Average daily precipitation on wet days', units='m')
    .to_dataset(name='PWA')
)
PWA_computed = PWA.compute()
PWA_computed

In [ ]:
ds_PA = PA
# sum((Pd - PA)^2) / (N - 1) per year per grid cell.
# .groupby('valid_time.year') groups days into annual blocks.
# The subtraction broadcasts PA (annual) back to daily frequency and then squares the result of the subtraction.
# Finally divide by the number of days in a year minus one, however:
# groupby.sum(skipna=True) returns 0 for all-NaN groups (ocean), so mask with PA.
# N is the actual number of days per year (365 or 366 for leap years).
days_per_year = ds['tot_prec'].groupby('valid_time.year').count()
PVAR = (
    (
        ((ds['tot_prec'].groupby('valid_time.year') - ds_PA['PA']) ** 2)
        .groupby('valid_time.year')
        .sum(skipna=True)
        / (days_per_year - 1)
    )
    .where(ds_PA['PA'].notnull())
).assign_attrs(long_name='Annual precipitation variance', units='m^2').to_dataset(name='PVAR')
PVAR_computed = PVAR.compute()
PVAR_computed

In [ ]:
ds_PWd = PW_d
ds_PWA = PWA
ds_W = W
# sum((PW_d - PWA)^2) / (W - 1) per year per grid cell.
# .groupby('valid_time.year') groups days into annual blocks.
# The subtraction broadcasts PWA (annual) back to daily frequency and then squares the result of the subtraction.
# skipna=True ignores dry days (NaN in PW_d), so only wet days contribute to the sum.
# Finally divide by (W - 1); mask with PWA which is NaN for both ocean cells and dry-year land cells.
PWVAR = (
    (
        ((ds_PWd['PW_d'].groupby('valid_time.year') - ds_PWA['PWA']) ** 2)
        .groupby('valid_time.year')
        .sum(skipna=True)
        / (ds_W['W'] - 1)
    )
    .where(ds_PWA['PWA'].notnull())
).assign_attrs(long_name='Annual wet day precipitation variance', units='m^2').to_dataset(name='PWVAR')
PWVAR_computed = PWVAR.compute()
PWVAR_computed

In [ ]:
ds_PW_pj = wet_days_perc

years = np.unique(ds['valid_time'].dt.year.values)
prec = ds['tot_prec']

# True for cells that have at least one non-NaN tot_prec value (i.e. land cells)
land_mask = prec.notnull().any(dim='valid_time')

# .where() masks days below the threshold to NaN, leaving only exceedance days.
# This produces NaN for both ocean cells and land cells with no exceedances in a year.
# min_count=0 resolves this by returning 0 for any all-NaN group; the land_mask
# then restores NaN specifically for ocean cells.
P95WT = (
    prec
    .where(prec >= ds_PW_pj['PW_95p'])
    .resample(valid_time='YE')
    .sum(dim='valid_time', skipna=True, min_count=0)
    .where(land_mask)
    .rename({'valid_time': 'year'})
    .rename('P95WT')
    .assign_coords(year=years)
    .assign_attrs(long_name='Total annual precipitation on very wet days (>= PW_95p)', units='m')
)
P95WT_computed = P95WT.compute()
P95WT_computed

In [ ]:
P99WT = (
    prec
    .where(prec >= ds_PW_pj['PW_99p'])
    .resample(valid_time='YE')
    .sum(dim='valid_time', skipna=True, min_count=0)
    .where(land_mask)
    .rename({'valid_time': 'year'})
    .rename('P99WT')
    .assign_coords(year=years)
    .assign_attrs(long_name='Total annual precipitation on extremely wet days (>= PW_99p)', units='m')
)
P99WT_computed = P95WT.compute()
P99WT_computed

In [ ]:
# Precipitation keeps Feb 29, so years are 365 or 366 days long.
# Rechunk with the actual per-year day counts so each block is exactly one year.
years_all = ds['valid_time'].dt.year.values
# Get an array of years present in the data and the number of days in each year
unique_years, year_lengths = np.unique(years_all, return_counts=True)
# Get the number of years
n_years = len(unique_years)
prec = ds['tot_prec'].chunk({'valid_time': year_lengths.tolist()})

# Thresholds are 2D (lat, lon) — rechunk to match prec spatial chunks so map_blocks aligns blocks correctly
pw95 = ds_PW_pj['PW_95p'].chunk({'latitude': prec.chunksizes['latitude'], 'longitude': prec.chunksizes['longitude']})
pw99 = ds_PW_pj['PW_99p'].chunk({'latitude': prec.chunksizes['latitude'], 'longitude': prec.chunksizes['longitude']})

# pw95_block/pw99_block (thresholds) and da_block (daily values to be evaluated) should be aligned
def process_block(da_block, pw95_block, pw99_block):
    # Wet day threshold: 1mm expressed in metres
    wet_threshold = 0.001
    # Extract precipitation and threshold arrays for this block
    p        = da_block.values              # (n_days, n_lat, n_lon)
    pw95_arr = pw95_block.values            # (n_lat, n_lon)
    pw99_arr = pw99_block.values            # (n_lat, n_lon)
    # Get the number of latitude and longitudes coordinates
    n_lat, n_lon = p.shape[1], p.shape[2]

    # Output arrays, initialised to NaN
    CDD   = np.full((1, n_lat, n_lon), np.nan, dtype=np.float32)  # longest dry spell (days)
    CWD   = np.full((1, n_lat, n_lon), np.nan, dtype=np.float32)  # longest wet spell (days)
    C95WD = np.full((1, n_lat, n_lon), np.nan, dtype=np.float32)  # longest very wet spell (days)
    C99WD = np.full((1, n_lat, n_lon), np.nan, dtype=np.float32)  # longest extremely wet spell (days)

    # Loop through each location
    for i in range(n_lat):
        for j in range(n_lon):
            # Get all the precipitation data throughout the year for that location
            p_1d = p[:, i, j]
            # If everything is NA (ocean cells) continue
            if np.all(np.isnan(p_1d)):
                continue
            # Loop over the four conditions, each writing into its own output array
            for cond, out in [
                (p_1d <  wet_threshold,   CDD  ),   # dry day
                (p_1d >= wet_threshold,   CWD  ),   # wet day
                (p_1d >  pw95_arr[i, j],  C95WD),   # very wet day
                (p_1d >  pw99_arr[i, j],  C99WD),   # extremely wet day
            ]:
                # Assign 0 if the condition is never met
                if not np.any(cond):
                    out[0, i, j] = 0
                    continue
                # Pad with zeros so diff detects runs that start on day 0 or end on the last day
                cond_int = cond.astype(np.int8)
                padded   = np.concatenate([[0], cond_int, [0]])
                # Take the diff between consecutive days, +1 marks the start of a spell; -1 marks the end
                d        = np.diff(padded)
                # Get all the start and end dates
                starts   = np.where(d ==  1)[0]
                ends     = np.where(d == -1)[0]
                # Get the length of each spell and keep the longest
                lengths  = ends - starts
                out[0, i, j] = lengths[np.argmax(lengths)]

    # Wrap results back into a Dataset with the first timestamp of the block as the time coordinate
    coords = {'valid_time': da_block['valid_time'].values[[0]],
              'latitude': da_block['latitude'].values,
              'longitude': da_block['longitude'].values}
    dims = ['valid_time', 'latitude', 'longitude']
    return xr.Dataset({
        'CDD':   xr.DataArray(CDD,   dims=dims, coords=coords),
        'CWD':   xr.DataArray(CWD,   dims=dims, coords=coords),
        'C95WD': xr.DataArray(C95WD, dims=dims, coords=coords),
        'C99WD': xr.DataArray(C99WD, dims=dims, coords=coords),
    })

# Get day-1 of the year coordinates (Jan 1 of each year, accounting for variable year lengths with Feb 29)
year_first_dates = prec['valid_time'].values[np.r_[0, year_lengths[:-1].cumsum()]]
# Gets spatial chunk sizes in main dataframe (to be given to the output df)
lat_chunks = prec.chunksizes['latitude']
lon_chunks = prec.chunksizes['longitude']
# Creates the empty (so far) output dataset with the correct structure
template = xr.Dataset({
    var: xr.DataArray(
        da.full(
            (n_years, len(prec['latitude']), len(prec['longitude'])),
            np.nan, chunks=(1, lat_chunks, lon_chunks), dtype=np.float32,
        ),
        dims=['valid_time', 'latitude', 'longitude'],
        coords={'valid_time': year_first_dates, 'latitude': prec['latitude'], 'longitude': prec['longitude']},
    )
    for var in ['CDD', 'CWD', 'C95WD', 'C99WD']
})

# Runs (lazily) the computations over each chunk
result = xr.map_blocks(process_block, prec, args=[pw95, pw99], template=template)

# Corrects and assigns the correct dimension
result = result.rename({'valid_time': 'year'}).assign_coords(year=unique_years)

# Assigns attributes to each variable
attrs = {
    'CDD':   ('Largest number of consecutive dry days (Pd < 1mm)', 'days'),
    'CWD':   ('Largest number of consecutive wet days (Pd >= 1mm)', 'days'),
    'C95WD': ('Largest number of consecutive very wet days (Pd > PW_95p)', 'days'),
    'C99WD': ('Largest number of consecutive extremely wet days (Pd > PW_99p)', 'days'),
}
for var, (long_name, unit) in attrs.items():
    result[var].attrs.update(long_name=long_name, units=unit)

result_computed = result.compute()
result_computed

In [ ]:
# Precipitation keeps Feb 29, so years are 365 or 366 days long.
# Rechunk with the actual per-year day counts so each block is exactly one year.
years_all = ds['valid_time'].dt.year.values
# Get an array of years present in the data and the number of days in each year
unique_years, year_lengths = np.unique(years_all, return_counts=True)
# Get the number of years
n_years = len(unique_years)
prec = ds['tot_prec'].chunk({'valid_time': year_lengths.tolist()})

# Thresholds are 2D (lat, lon) — rechunk to match prec spatial chunks so map_blocks aligns blocks correctly
pw95 = ds_PW_pj['PW_95p'].chunk({'latitude': prec.chunksizes['latitude'], 'longitude': prec.chunksizes['longitude']})
pw99 = ds_PW_pj['PW_99p'].chunk({'latitude': prec.chunksizes['latitude'], 'longitude': prec.chunksizes['longitude']})

# pw95_block/pw99_block (thresholds) and da_block (daily values to be evaluated) should be aligned
def process_block(da_block, pw95_block, pw99_block):
    # Wet day threshold: 1mm expressed in metres
    wet_threshold = 0.001
    # Extract precipitation and threshold arrays for this block
    p        = da_block.values              # (n_days, n_lat, n_lon)
    pw95_arr = pw95_block.values            # (n_lat, n_lon)
    pw99_arr = pw99_block.values            # (n_lat, n_lon)
    # Get the number of latitude and longitudes coordinates
    n_lat, n_lon = p.shape[1], p.shape[2]

    # Output arrays, initialised to NaN
    PCWD   = np.full((1, n_lat, n_lon), np.nan, dtype=np.float32)  # precip total in longest wet spell
    PC95WD = np.full((1, n_lat, n_lon), np.nan, dtype=np.float32)  # precip total in longest very wet spell
    PC99WD = np.full((1, n_lat, n_lon), np.nan, dtype=np.float32)  # precip total in longest extremely wet spell

    # Loop through each location
    for i in range(n_lat):
        for j in range(n_lon):
            # Get all the precipitation data throughout the year for that location
            p_1d = p[:, i, j]
            # If everything is NA (ocean cells) continue
            if np.all(np.isnan(p_1d)):
                continue
            # Loop over the three conditions, each writing into its own output array
            for cond, out in [
                (p_1d >= wet_threshold,  PCWD  ),   # wet day
                (p_1d >  pw95_arr[i, j], PC95WD),   # very wet day
                (p_1d >  pw99_arr[i, j], PC99WD),   # extremely wet day
            ]:
                # Assign 0 if the condition is never met
                if not np.any(cond):
                    out[0, i, j] = 0.0
                    continue
                # Pad with zeros so diff detects runs that start on day 0 or end on the last day
                cond_int = cond.astype(np.int8)
                padded   = np.concatenate([[0], cond_int, [0]])
                # Take the diff between consecutive days, +1 marks the start of a spell; -1 marks the end
                d        = np.diff(padded)
                # Get all the start and end dates
                starts   = np.where(d ==  1)[0]
                ends     = np.where(d == -1)[0]
                # Find the longest spell and sum precipitation over it
                lengths  = ends - starts
                k        = np.argmax(lengths)
                out[0, i, j] = float(np.nansum(p_1d[starts[k]:ends[k]]))

    # Wrap results back into a Dataset with the first timestamp of the block as the time coordinate
    coords = {'valid_time': da_block['valid_time'].values[[0]],
              'latitude': da_block['latitude'].values,
              'longitude': da_block['longitude'].values}
    dims = ['valid_time', 'latitude', 'longitude']
    return xr.Dataset({
        'PCWD':   xr.DataArray(PCWD,   dims=dims, coords=coords),
        'PC95WD': xr.DataArray(PC95WD, dims=dims, coords=coords),
        'PC99WD': xr.DataArray(PC99WD, dims=dims, coords=coords),
    })

# Get day-1 of the year coordinates (Jan 1 of each year, accounting for variable year lengths with Feb 29)
year_first_dates = prec['valid_time'].values[np.r_[0, year_lengths[:-1].cumsum()]]
# Gets spatial chunk sizes in main dataframe (to be given to the output df)
lat_chunks = prec.chunksizes['latitude']
lon_chunks = prec.chunksizes['longitude']
# Creates the empty (so far) output dataset with the correct structure
template = xr.Dataset({
    var: xr.DataArray(
        da.full(
            (n_years, len(prec['latitude']), len(prec['longitude'])),
            np.nan, chunks=(1, lat_chunks, lon_chunks), dtype=np.float32,
        ),
        dims=['valid_time', 'latitude', 'longitude'],
        coords={'valid_time': year_first_dates, 'latitude': prec['latitude'], 'longitude': prec['longitude']},
    )
    for var in ['PCWD', 'PC95WD', 'PC99WD']
})

# Runs (lazily) the computations over each chunk
result = xr.map_blocks(process_block, prec, args=[pw95, pw99], template=template)

# Corrects and assigns the correct dimension
result = result.rename({'valid_time': 'year'}).assign_coords(year=unique_years)

# Assigns attributes to each variable
attrs = {
    'PCWD':   ('Total precipitation during longest consecutive wet day period', 'm'),
    'PC95WD': ('Total precipitation during longest consecutive very wet day period (> PW_95p)', 'm'),
    'PC99WD': ('Total precipitation during longest consecutive extremely wet day period (> PW_99p)', 'm'),
}
for var, (long_name, unit) in attrs.items():
    result[var].attrs.update(long_name=long_name, units=unit)

result_computed = result.compute()
result_computed

In [ ]:
years = np.unique(ds['valid_time'].dt.year.values)
prec = ds['tot_prec']

PX1 = (
    prec
    .resample(valid_time='YE')
    .max(dim='valid_time', skipna=True)
    .rename({'valid_time': 'year'})
    .rename('PX1')
    .assign_coords(year=years)
    .assign_attrs(long_name='Annual maximum 1-day precipitation', units='m')
)
PX1_computed = PX1.compute()
PX1_computed

In [ ]:
PX5 = (
    prec
    .rolling(valid_time=5, min_periods=5)
    .sum()
    .resample(valid_time='YE')
    .max(dim='valid_time', skipna=True)
    .rename({'valid_time': 'year'})
    .rename('PX5')
    .assign_coords(year=years)
    .assign_attrs(long_name='Annual maximum 5-day precipitation total', units='m')
)
PX5_computed = PX5.compute()
PX5_computed

In [ ]:
ds_P_jm = P_jm
PXM = (
    ds_P_jm['P_jm']
    .resample(valid_time='YE')
    .max(dim='valid_time', skipna=True)
    .rename({'valid_time': 'year'})
    .rename('PXM')
    .assign_coords(year=np.unique(ds_P_jm['valid_time'].dt.year.values))
    .assign_attrs(long_name='Maximum monthly total precipitation', units='m')
)
PXM_computed = PXM.compute()
PXM_computed

In [ ]:
PNM = (
    ds_P_jm['P_jm']
    .resample(valid_time='YE')
    .min(dim='valid_time', skipna=True)
    .rename({'valid_time': 'year'})
    .rename('PNM')
    .assign_coords(year=np.unique(ds_P_jm['valid_time'].dt.year.values))
    .assign_attrs(long_name='Minimum monthly total precipitation', units='m')
)
PNM_computed = PNM.compute()
PNM_computed